In [0]:
from pyspark.sql.functions import col, current_timestamp

# 1. READ the Silver Tables
drivers_df = spark.read.table("workspace.default.silver_drivers") \
    .withColumnRenamed("name", "driver_name") \
    .withColumnRenamed("nationality", "driver_nationality") \
    # .withColumnRenamed("number", "driver_number") # <--- Commenting out the suspect line

constructors_df = spark.read.table("workspace.default.silver_constructors") \
    .withColumnRenamed("name", "team") 

races_df = spark.read.table("workspace.default.silver_races") \
    .withColumnRenamed("name", "race_name") \
    .withColumnRenamed("race_timestamp", "race_date") 

results_df = spark.read.table("workspace.default.silver_results") \
    .withColumnRenamed("time", "race_time")

# 2. JOIN
race_results_df = results_df \
    .join(races_df, results_df.race_id == races_df.race_id) \
    .join(drivers_df, results_df.driver_id == drivers_df.driver_id) \
    .join(constructors_df, results_df.constructor_id == constructors_df.constructor_id)

# 3. SELECT
final_df = race_results_df.select(
    col("race_year"),
    col("race_name"),
    col("race_date"),
    col("circuit_id"),
    col("driver_name"),
    # col("driver_number"), # <--- Commenting out the problem here too
    col("driver_nationality"),
    col("team"),
    col("grid"),
    col("fastest_lap"),
    col("race_time"),
    col("points"),
    col("position")
).withColumn("created_date", current_timestamp())

# 4. VERIFY
display(final_df)

# Save the Gold Table
final_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_race_results")

print("Gold Table Saved Successfully! 🏆")

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, to_timestamp, concat, lit, current_timestamp, when

# 1. READ
races_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/raw_data/raw_data/races.csv")

# 2. TRANSFORM
# Use .printSchema for the headers
races_silver_df = races_df \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("year", "race_year") \
    .withColumnRenamed("round", "race_round") \
    .withColumnRenamed("circuitId", "circuit_id") \
    .withColumn("time", when(col("time") == "\\N", lit("00:00:00")).otherwise(col("time"))) \
    .withColumn("race_timestamp", to_timestamp(concat(col("date"), lit(" "), col("time")), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("ingestion_date", current_timestamp()) \
    .select("race_id", "race_year", "race_round", "circuit_id", "name", "race_timestamp", "ingestion_date")

# 3. WRITE
races_silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_races")

display(races_silver_df)

In [0]:
from pyspark.sql.functions import sum, count, col, when, desc

# 1. READ the Gold Table
race_results_df = spark.read.table("workspace.default.gold_race_results")

# 2. AGGREGATE 
# FIX: Notice we use "1" (with quotes) to handle the text data safely
driver_standings_df = race_results_df \
    .groupBy("race_year", "driver_name", "driver_nationality", "team") \
    .agg(
        sum("points").alias("total_points"),
        count(when(col("position") == "1", True)).alias("wins") 
    ) \
    .orderBy(desc("total_points"))

# 3. WRITE
driver_standings_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_driver_standings")

display(driver_standings_df)

In [0]:
%sql
SELECT 
    driver_name,
    SUM(wins) as total_wins,
    SUM(total_points) as career_points
FROM workspace.default.gold_driver_standings
GROUP BY driver_name
ORDER BY total_wins DESC
LIMIT 5;

In [0]:
driver_standings_df.printSchema()

In [0]:
from pyspark.sql.functions import sum as _sum

# This creates the 'total_points' column Spark was looking for
final_df = final_df.groupBy("driver_name", "race_year", "team") \
                   .agg(_sum("points").alias("total_points"))

In [0]:
# Restore final_df to its original definition from cell 1
from pyspark.sql.functions import col, current_timestamp

# 1. READ the Silver Tables
drivers_df = spark.read.table("workspace.default.silver_drivers") \
    .withColumnRenamed("name", "driver_name") \
    .withColumnRenamed("nationality", "driver_nationality")

constructors_df = spark.read.table("workspace.default.silver_constructors") \
    .withColumnRenamed("name", "team")

races_df = spark.read.table("workspace.default.silver_races") \
    .withColumnRenamed("name", "race_name") \
    .withColumnRenamed("race_timestamp", "race_date")

results_df = spark.read.table("workspace.default.silver_results") \
    .withColumnRenamed("time", "race_time")

# 2. JOIN
race_results_df = results_df \
    .join(races_df, results_df.race_id == races_df.race_id) \
    .join(drivers_df, results_df.driver_id == drivers_df.driver_id) \
    .join(constructors_df, results_df.constructor_id == constructors_df.constructor_id)

# 3. SELECT
final_df = race_results_df.select(
    col("race_year"),
    col("race_name"),
    col("race_date"),
    col("circuit_id"),
    col("driver_name"),
    col("driver_nationality"),
    col("team"),
    col("grid"),
    col("fastest_lap"),
    col("race_time"),
    col("points"),
    col("position")
).withColumn("created_date", current_timestamp())

display(final_df)

In [0]:
from pyspark.sql.functions import expr, col, regexp_replace

# 1. Turn off ANSI mode
spark.conf.set("spark.sql.ansi.enabled", "false")

# 2. Clean '\\N', empty strings, and whitespace from columns before casting
final_df = final_df.withColumn("position", regexp_replace(col("position"), "^\\\\N$|^$|^\\s+$", None)) \
                   .withColumn("race_year", regexp_replace(col("race_year"), "^\\\\N$|^$|^\\s+$", None)) \
                   .withColumn("points", regexp_replace(col("points"), "^\\\\N$|^$|^\\s+$", None))

final_df = final_df.withColumn("position", expr("try_cast(position AS INT)")) \
                   .withColumn("race_year", expr("try_cast(race_year AS INT)")) \
                   .withColumn("points", expr("try_cast(points AS FLOAT)"))

display(final_df)

# 3. Create Database & Save
spark.sql("CREATE DATABASE IF NOT EXISTS f1_presentation")
final_df.write.mode("overwrite").format("delta").saveAsTable("f1_presentation.gold_race_results")
print("SUCCESS! The table has been saved to the Gold layer.")

In [0]:
%sql
SELECT * FROM f1_presentation.gold_race_results
ORDER BY points DESC 
LIMIT 5